# OCR Lead notebook v3 (Person 2)
**Part A** - benchmark the engines on val and pick the winner. **Part B** - run the winner on train / val / test, saved to Drive.

**Three separate sessions, because the engines need different libraries:**
- **Session 1 (`MODE = "main"`)**: Tesseract + EasyOCR. This is also where the final run happens if one of them wins.
- **Session 2 (`MODE = "paddle"`)**: a *fresh* session for Paddle (GPU). Paddle's install overwrites NVIDIA libraries that PyTorch (EasyOCR) needs, so the two must **never** share a session. This session removes PyTorch.
- **Session 3 (`MODE = "surya"`)**: a *fresh* session for Surya 0.17.1, which pins older package versions.

In sessions 2 and 3 run down to the benchmark cell, then stop. Their results are saved to Drive, and the results cell in any session reads them.
If Paddle or Surya wins, run Part B in that session (the notebook tells you if you are in the wrong one).

Runtime > Change runtime type > GPU, in every session.

In [ ]:
# ===== CONFIG (edit only this cell) =====
MODE = "surya"            # "main"   -> tesseract + easyocr
                         # "paddle" -> FRESH session: Paddle GPU (removes PyTorch, so never share it with EasyOCR)
                         # "surya"  -> FRESH session: surya-ocr 0.17.1
SAMPLE_PER_GROUP = 15    # docs per (language, quality) group on val; None = use the full val
SKIP_INSTALL = False # set True after a runtime restart (packages are already installed)
FORCE_PADDLE_CPU = False # True = test Paddle even with the CPU build only (about 20 s/doc, very slow)

ENGINES_BY_MODE = {"main": ["tesseract", "easyocr"], "paddle": ["paddle"], "surya": ["surya"]}
assert MODE in ENGINES_BY_MODE, "MODE must be 'main', 'paddle' or 'surya'"
ENGINES_TO_TEST = list(ENGINES_BY_MODE[MODE])
print(MODE, ENGINES_TO_TEST)

surya ['surya']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, random, shutil, re, unicodedata, sys, subprocess
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

BASE = "/content/drive/MyDrive/OCR dataset v1"
DATA_DIR = BASE + "/data"
PROCESSED_IMAGES = DATA_DIR + "/processed/images"
SYNTH_IMAGES = DATA_DIR + "/synth/out/images"
SPLITS_DIR = DATA_DIR + "/splits"

OUT_DIR = BASE + "/ocr_output"        # final *_with_ocr.jsonl (on Drive)
BENCH_DIR = BASE + "/ocr_benchmark"   # benchmark csv files    (on Drive)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BENCH_DIR, exist_ok=True)

for s in ["train", "val", "test"]:
    print(s, os.path.exists(f"{SPLITS_DIR}/{s}.jsonl"))
print("processed images:", len(os.listdir(PROCESSED_IMAGES)))
print("synthetic images:", len(os.listdir(SYNTH_IMAGES)))

Mounted at /content/drive
train True
val True
test True
processed images: 1000
synthetic images: 3500


## Install
Each mode installs only what it needs. In `paddle` mode the Paddle GPU package comes from Paddle's own index (it is not on normal pip), and it can take 5 to 15 minutes.
If Colab asks you to restart the session after this cell: restart, set `SKIP_INSTALL = True` in the config cell, and run again from the top.

In [ ]:
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    ok = r.returncode == 0
    print(("OK   " if ok else "FAIL ") + cmd[:100])
    if not ok:
        print("\n".join((r.stdout + r.stderr).strip().splitlines()[-4:]))
    return ok

if SKIP_INSTALL:
    print("install skipped")
elif MODE == "main":
    sh("apt-get -qq install -y tesseract-ocr tesseract-ocr-ara tesseract-ocr-eng")
    sh("pip -q install pytesseract easyocr jiwer rapidfuzz")
elif MODE == "paddle":
    sh("pip -q install jiwer rapidfuzz")
    sh("pip -q uninstall -y torch torchvision torchaudio easyocr")   # Paddle's NVIDIA libraries clash with PyTorch; Paddle does not need it
    sh("pip -q uninstall -y paddlepaddle paddlepaddle-gpu")
    gpu_ok = sh("pip -q install paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/")
    if not gpu_ok:
        print("GPU Paddle could not be installed (maybe no wheel for this Python version).")
        if FORCE_PADDLE_CPU:
            sh("pip -q install paddlepaddle")
    if gpu_ok or FORCE_PADDLE_CPU:
        sh("pip -q install paddleocr")
else:   # surya session
    sh("pip -q install jiwer rapidfuzz")
    sh('pip -q install "surya-ocr==0.17.1"')
    sh('pip -q install "transformers==4.57.3"')   # Surya 0.17.1 breaks on newer transformers (pad_token_id error); restart the runtime after this

OK   pip -q install jiwer rapidfuzz
OK   pip -q install "surya-ocr==0.17.1"
OK   pip -q install "transformers==4.57.3"


In [ ]:
%%writefile engines.py
"""
ocr/engines.py  (Person 2)  -- v2

One wrapper per OCR engine, all returning the same output:
    [{"text": str, "bbox": [x1, y1, x2, y2], "conf": float 0..1}, ...]

This matches the "ocr.words" field of the team data contract.

Usage:
    from ocr.engines import run_ocr, ENGINES
    words = run_ocr("tesseract", "data/processed/images/cord_0001.jpg")
    words, seconds = run_ocr("paddle", path, timed=True)

Granularity differs per engine:
    tesseract -> words      easyocr / paddle / surya -> text lines (or blocks for surya >= 0.20)

Heavy libraries are imported lazily, so a missing engine only fails
when you actually call it.

v2 changes:
    - Paddle: oneDNN (mkldnn) turned off, which avoids the NotImplementedError on paddle 3.x
    - Surya: works with both the old API (surya-ocr 0.17.x) and the new one (0.20+)
"""
import re
import time
from functools import lru_cache

Word = dict  # {"text": str, "bbox": [x1,y1,x2,y2], "conf": float}


# ---------- helpers ----------
def _poly_to_bbox(poly):
    """Convert a polygon [[x,y],...] to [x1,y1,x2,y2] with ints."""
    xs = [float(p[0]) for p in poly]
    ys = [float(p[1]) for p in poly]
    return [int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))]


def _clamp01(x):
    try:
        return max(0.0, min(1.0, float(x)))
    except (TypeError, ValueError):
        return 0.0


def _word(text, bbox, conf):
    return {"text": str(text).strip(), "bbox": [int(v) for v in bbox], "conf": _clamp01(conf)}


def _clean(words):
    return [w for w in words if w["text"]]


def _strip_tags(s):
    """Remove simple HTML tags (<b>, <br>, ...) and squeeze whitespace."""
    s = re.sub(r"<[^>]+>", " ", s or "")
    return re.sub(r"\s+", " ", s).strip()


# ---------- Tesseract ----------
def ocr_tesseract(image_path, lang="ara+eng", psm=6) -> list[Word]:
    import pytesseract
    from PIL import Image

    img = Image.open(image_path).convert("RGB")
    data = pytesseract.image_to_data(
        img, lang=lang, config=f"--psm {psm}", output_type=pytesseract.Output.DICT
    )
    words = []
    for i, txt in enumerate(data["text"]):
        try:
            conf = float(data["conf"][i])
        except (TypeError, ValueError):
            continue
        if conf < 0 or not str(txt).strip():
            continue  # -1 means "not a word" (layout block)
        x, y, w, h = (data[k][i] for k in ("left", "top", "width", "height"))
        words.append(_word(txt, [x, y, x + w, y + h], conf / 100.0))
    return _clean(words)


# ---------- EasyOCR ----------
@lru_cache(maxsize=2)
def _easyocr_reader(langs=("ar", "en"), gpu=True):
    import easyocr
    return easyocr.Reader(list(langs), gpu=gpu)


def ocr_easyocr(image_path, langs=("ar", "en"), gpu=True) -> list[Word]:
    reader = _easyocr_reader(tuple(langs), gpu)
    results = reader.readtext(str(image_path))  # [(poly, text, conf), ...]
    return _clean([_word(t, _poly_to_bbox(p), c) for p, t, c in results])


# ---------- PaddleOCR (paddle 3.x and 2.x) ----------
@lru_cache(maxsize=2)
def _paddle_engine(lang="ar"):
    from paddleocr import PaddleOCR

    v3 = dict(lang=lang, use_doc_orientation_classify=False,
              use_doc_unwarping=False, use_textline_orientation=False)
    attempts = [
        dict(v3, enable_mkldnn=False),            # 3.x, oneDNN off (fixes the NotImplementedError)
        v3,                                        # 3.x without the flag
        dict(lang=lang, use_angle_cls=True, show_log=False),  # 2.x
    ]
    last = None
    for kwargs in attempts:
        try:
            return PaddleOCR(**kwargs)
        except (TypeError, ValueError) as e:
            last = e
    raise last


def ocr_paddle(image_path, lang="ar") -> list[Word]:
    engine = _paddle_engine(lang)
    words = []
    if hasattr(engine, "predict"):  # v3.x
        for res in engine.predict(str(image_path)):
            data = getattr(res, "json", None)
            r = data["res"] if isinstance(data, dict) and "res" in data else res
            texts = r.get("rec_texts", [])
            scores = r.get("rec_scores", [])
            polys = r.get("rec_polys", r.get("dt_polys", []))
            for t, s, p in zip(texts, scores, polys):
                words.append(_word(t, _poly_to_bbox(p), s))
    else:  # v2.x
        out = engine.ocr(str(image_path), cls=True)
        for page in out or []:
            for poly, (t, s) in page or []:
                words.append(_word(t, _poly_to_bbox(poly), s))
    return _clean(words)


# ---------- Surya ----------
# surya-ocr 0.17.x  ("v1"): FoundationPredictor + DetectionPredictor, output has text_lines
# surya-ocr 0.20+  ("v2"): SuryaInferenceManager (starts a model server), output has blocks with html
@lru_cache(maxsize=1)
def _surya_api():
    try:
        from surya.inference import SuryaInferenceManager  # only exists in 0.20+
        from surya.recognition import RecognitionPredictor
        return "v2", RecognitionPredictor(SuryaInferenceManager()), None
    except ImportError:
        pass
    from surya.detection import DetectionPredictor
    from surya.foundation import FoundationPredictor
    from surya.recognition import RecognitionPredictor
    return "v1", RecognitionPredictor(FoundationPredictor()), DetectionPredictor()


def ocr_surya(image_path) -> list[Word]:
    from PIL import Image

    version, rec, det = _surya_api()
    img = Image.open(image_path).convert("RGB")
    words = []
    if version == "v2":
        page = rec([img])[0]
        for b in page.blocks:
            words.append(_word(_strip_tags(b.html), b.bbox, getattr(b, "confidence", 0.0)))
    else:
        result = rec([img], det_predictor=det)[0]
        for line in result.text_lines:
            words.append(_word(_strip_tags(line.text), line.bbox, getattr(line, "confidence", 0.0)))
    return _clean(words)


# ---------- registry ----------
ENGINES = {
    "tesseract": ocr_tesseract,
    "easyocr": ocr_easyocr,
    "paddle": ocr_paddle,
    "surya": ocr_surya,
}


def run_ocr(engine: str, image_path, timed=False, **kwargs):
    """Run one engine. If timed=True returns (words, seconds)."""
    if engine not in ENGINES:
        raise ValueError(f"Unknown engine '{engine}'. Choose from {list(ENGINES)}")
    t0 = time.perf_counter()
    words = ENGINES[engine](image_path, **kwargs)
    dt = time.perf_counter() - t0
    return (words, dt) if timed else words


def words_to_text(words: list[Word]) -> str:
    """Join words into reading order (top-to-bottom, then left-to-right).
    For RTL-heavy lines you may want to sort right-to-left instead."""
    ordered = sorted(words, key=lambda w: (round(w["bbox"][1] / 15), w["bbox"][0]))
    return " ".join(w["text"] for w in ordered)


if __name__ == "__main__":
    import json
    import sys

    path = sys.argv[1]
    for name in ENGINES:
        try:
            w, dt = run_ocr(name, path, timed=True)
            print(f"{name}: {len(w)} items, {dt:.2f}s")
            print(json.dumps(w[:3], ensure_ascii=False))
        except Exception as e:  # one broken engine must not stop the others
            print(f"{name}: FAILED -> {type(e).__name__}: {e}")


Writing engines.py


In [ ]:
sys.path.insert(0, "/content")
from engines import run_ocr, words_to_text, ENGINES
print("engines:", list(ENGINES))

PADDLE_GPU = None
if "paddle" in ENGINES_TO_TEST:
    try:
        import paddle
        PADDLE_GPU = bool(paddle.device.is_compiled_with_cuda())
        print("paddle", paddle.__version__, "| GPU build:", PADDLE_GPU)
    except Exception as e:
        print("paddle is not importable:", type(e).__name__, e)
    if PADDLE_GPU is None:
        ENGINES_TO_TEST.remove("paddle")
        print("-> Paddle removed from this run")
    elif not PADDLE_GPU and not FORCE_PADDLE_CPU:
        ENGINES_TO_TEST.remove("paddle")
        print("-> Paddle removed: only the CPU build is installed (about 20 s/doc). Set FORCE_PADDLE_CPU = True to test it anyway.")

def label_of(engine):
    """Name used in the result files and tables (separates GPU from CPU Paddle)."""
    if engine == "paddle":
        return "paddle_gpu" if PADDLE_GPU else "paddle_cpu"
    return engine

ENGINE_OF = {"paddle_gpu": "paddle", "paddle_cpu": "paddle"}
print("will benchmark:", ENGINES_TO_TEST)

if MODE == "surya":
    import transformers
    print("transformers", transformers.__version__)
    assert transformers.__version__ == "4.57.3", (
        "Surya 0.17.1 needs transformers==4.57.3 (you have " + transformers.__version__ + "). "
        "Run the install cell, then Runtime > Restart session, set SKIP_INSTALL = True and run again.")

    import torch, torchvision
    from torchvision.ops import nms
    try:
        nms(torch.zeros((1, 4)), torch.zeros(1), 0.5)
        print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| nms OK")
    except Exception as e:
        raise RuntimeError("torch and torchvision do not match (" + str(e) + "). "
                           "Runtime > Disconnect and delete runtime, then start again without touching torch.")


engines: ['tesseract', 'easyocr', 'paddle', 'surya']
will benchmark: ['surya']
transformers 4.57.3
torch 2.14.1+cpu | torchvision 0.29.1+cpu | nms OK


## Helpers (data loading and scoring)

In [ ]:
import jiwer
from rapidfuzz import fuzz

FIELDS = ["invoice_number", "date", "supplier", "total"]
CORE_FIELDS = ["invoice_number", "supplier", "total"]   # date is excluded from the main score: its printed format rarely matches the label

def load_split(split):
    with open(f"{SPLITS_DIR}/{split}.jsonl", encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

import os

def resolve_image_path(image_path):
    if os.path.exists(image_path):
        return image_path

    if image_path.startswith("data/"):
        image_path = image_path[len("data/"):]

    return os.path.join(DATA_DIR, image_path)

AR_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")

def norm(s):
    s = unicodedata.normalize("NFKC", str(s or "")).translate(AR_DIGITS)
    s = re.sub(r"[ً-ٰٟـ]", "", s)   # diacritics + tatweel
    s = re.sub("[آإأ]", "ا", s)
    s = s.replace("ى", "ي").replace("ة", "ه")
    return s.lower()

def norm_text(s):
    return re.sub(r"\s+", " ", norm(s)).strip()

def norm_compact(s):
    return re.sub(r"[\W_]+", "", norm(s))

def get_ref_text(rec):
    # full transcription of the document, if it exists in the record
    gt = rec.get("ground_truth", {}) or {}
    for k in ("full_text", "text", "transcription", "ocr_text"):
        for holder in (rec, gt):
            v = holder.get(k) if isinstance(holder, dict) else None
            if isinstance(v, str) and v.strip():
                return v
    return None

def field_hits(rec, hyp_text):
    h = norm_compact(hyp_text)
    out = {}
    for f in FIELDS:
        v = norm_compact((rec.get("ground_truth", {}) or {}).get(f, ""))
        if not v:
            out["hit_" + f] = None
        elif v in h:
            out["hit_" + f] = 1.0
        elif len(v) >= 5 and fuzz.partial_ratio(v, h) >= 90:
            out["hit_" + f] = 1.0
        else:
            out["hit_" + f] = 0.0
    def mean_of(names):
        vals = [out["hit_" + n] for n in names if out["hit_" + n] is not None]
        return sum(vals) / len(vals) if vals else None
    out["field_hit"] = mean_of(FIELDS)
    out["field_hit_core"] = mean_of(CORE_FIELDS)
    return out

def score_doc(rec, words):
    hyp = words_to_text(words)
    res = field_hits(rec, hyp)
    ref = get_ref_text(rec)
    if ref:
        r, h = norm_text(ref), norm_text(hyp)
        res["cer"] = jiwer.cer(r, h) if r else None
        res["wer"] = jiwer.wer(r, h) if r else None
    else:
        res["cer"] = res["wer"] = None
    return res

val_all = load_split("val")
print("val size:", len(val_all))
print("has full text for CER/WER:", get_ref_text(val_all[0]) is not None)
print(pd.Series([(r["source"], r["language"], r["quality"]) for r in val_all]).value_counts())

val size: 516
has full text for CER/WER: False
(cord, en, clear)             149
(synthetic, ar, clear)         73
(synthetic, ar, tilted)        48
(synthetic, ar, noisy)         40
(synthetic, mixed, clear)      39
(synthetic, ar, blurry)        36
(synthetic, mixed, noisy)      32
(synthetic, mixed, blurry)     29
(synthetic, en, clear)         22
(synthetic, mixed, tilted)     19
(synthetic, en, tilted)        14
(synthetic, en, blurry)         9
(synthetic, en, noisy)          6
Name: count, dtype: int64


In [ ]:
import json

path = f"{OUT_DIR}/train_with_ocr.jsonl"
checked = 0
found = 0

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)

        if row.get("ocr", {}).get("error") == "Image not found":
            resolved = resolve_image_path(row["image_path"])
            checked += 1
            found += os.path.exists(resolved)

            if checked <= 5:
                print(row["doc_id"], "->", os.path.exists(resolved))

print(f"\nFound: {found}/{checked}")


Found: 0/0


## Part A - Benchmark on val
Without a full-text field, engines are ranked by `field_hit_core`: the share of the labelled fields (invoice number, supplier, total) found inside the OCR text.
Results of every engine are saved on Drive, one CSV per engine, and a re-run resumes where it stopped.

In [ ]:
def sample_records(recs, per_group):
    if not per_group:
        return recs
    random.seed(0)
    groups = {}
    for r in recs:
        groups.setdefault((r["language"], r["quality"]), []).append(r)
    out = []
    for g in groups.values():
        out += random.sample(g, min(per_group, len(g)))
    return out

bench_recs = sample_records(val_all, SAMPLE_PER_GROUP)
print("benchmark documents:", len(bench_recs))

benchmark documents: 164


In [ ]:
def bench_engine(engine, recs):
    label = label_of(engine)
    csv_path = f"{BENCH_DIR}/{label}_val.csv"
    done = pd.read_csv(csv_path) if os.path.exists(csv_path) else pd.DataFrame()
    done_ids = set(done["doc_id"]) if len(done) else set()
    rows = done.to_dict("records")

    # warm-up: loads the model and checks the engine works at all (not timed)
    first = next((resolve_image_path(r["image_path"]) for r in recs if resolve_image_path(r["image_path"])), None)
    if first is None:
        print("!! no benchmark image found, check the paths")
        return pd.DataFrame(rows)
    try:
        run_ocr(engine, first)
    except Exception as e:
        print(f"!! {label} could not run: {type(e).__name__}: {e}")
        return pd.DataFrame(rows)

    for i, rec in enumerate(tqdm(recs, desc=label)):
        if rec["doc_id"] in done_ids:
            continue
        path = resolve_image_path(rec["image_path"])
        if path is None:
            continue
        try:
            words, secs = run_ocr(engine, path, timed=True)
            row = score_doc(rec, words)
            row["n_words"] = len(words)
            row["error"] = ""
        except Exception as e:
            secs, row = None, {"error": str(e)[:200]}
        row.update(engine=label, doc_id=rec["doc_id"], language=rec["language"],
                   quality=rec["quality"], source=rec["source"], time=secs)
        rows.append(row)
        if i % 20 == 0:
            pd.DataFrame(rows).to_csv(csv_path, index=False)
    df = pd.DataFrame(rows)
    df.to_csv(csv_path, index=False)
    return df

for eng in ENGINES_TO_TEST:
    d = bench_engine(eng, bench_recs)
    print(label_of(eng), "->", len(d), "documents saved")










Recognizing Text: 100%|██████████| 22/22 [00:07<00:00,  3.09it/s]


surya:   0%|          | 0/164 [00:00<?, ?it/s]


Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  3.79it/s]

Recognizing Text: 100%|██████████| 22/22 [00:01<00:00, 11.72it/s]

Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  5.18it/s]

Recognizing Text: 100%|██████████| 30/30 [00:01<00:00, 16.89it/s]

Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  4.99it/s]

Recognizing Text: 100%|██████████| 46/46 [00:02<00:00, 22.21it/s]

Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  5.33it/s]

Recognizing Text: 100%|██████████| 22/22 [00:00<00:00, 22.04it/s]

Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  4.60it/s]

Recognizing Text: 100%|██████████| 30/30 [00:01<00:00, 18.33it/s]

Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  5.21it/s]

Recognizing Text: 100%|██████████| 30/30 [00:01<00:00, 15.37it/s]

Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  3.89it/s]

Recognizing Text: 100%|██████████| 38/38 [00:02<00:00, 18.12it/s]

Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  5.47it/s]

Recognizin

surya -> 164 documents saved


In [ ]:
# DISABLED: force-reinstalling transformers also reinstalls its dependencies and can break torch.
# The install cell (4) already pins transformers==4.57.3 in surya mode. Restart the session after the install cell.
# %pip install --force-reinstall "transformers==4.57.3"


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Available:", torch.cuda.is_available())

PyTorch: 2.14.1+cpu
CUDA: None
Available: False


In [ ]:
# DISABLED: reinstalling torch unpinned mismatches it with torchvision/Surya
# ('operator torchvision::nms does not exist'). If that error appears, use a fresh runtime instead.
# %pip uninstall -y torch torchvision torchaudio
# %pip install torch torchvision torchaudio


## Results
Reads **every** engine's CSV from Drive (including the Surya session), and compares only the documents that all engines scored.
If one engine has far fewer documents than the rest, finish it first, or the comparison shrinks to that small set.

In [ ]:
ALL_LABELS = ["tesseract", "easyocr", "paddle", "surya"]
frames = {}
for lab in ALL_LABELS:
    p = f"{BENCH_DIR}/{lab}_val.csv"
    if os.path.exists(p):
        d = pd.read_csv(p)
        if len(d):
            frames[lab] = d
assert frames, "No benchmark results found in " + BENCH_DIR
print("documents per engine:", {k: len(v) for k, v in frames.items()})

common = set.intersection(*[set(d["doc_id"]) for d in frames.values()])
print("documents scored by ALL engines:", len(common))
assert common, "No document was scored by all engines. Finish or remove the incomplete engine's CSV."

all_df = pd.concat(frames.values(), ignore_index=True)
all_df = all_df[all_df["doc_id"].isin(common)].copy()
for c in ["cer", "wer", "field_hit", "field_hit_core", "time"]:
    if c not in all_df:
        all_df[c] = np.nan

# recompute the scores from the per-field hits, so result files saved by the older notebook also work
for f in FIELDS:
    if "hit_" + f not in all_df:
        all_df["hit_" + f] = np.nan
all_df["field_hit"] = all_df[["hit_" + f for f in FIELDS]].mean(axis=1, skipna=True)
all_df["field_hit_core"] = all_df[["hit_" + f for f in CORE_FIELDS]].mean(axis=1, skipna=True)

def boot_ci(x, n=2000, seed=0):
    x = np.array([v for v in x if pd.notna(v)], dtype=float)
    if len(x) == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    means = rng.choice(x, size=(n, len(x)), replace=True).mean(axis=1)
    return (float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5)))

metrics = ["cer", "wer", "field_hit_core", "field_hit", "time"]

summary = all_df.groupby("engine")[metrics].mean().round(3)
cis = {e: boot_ci(g["field_hit_core"]) for e, g in all_df.groupby("engine")}
summary["core_ci95_low"] = [round(cis[e][0], 3) for e in summary.index]
summary["core_ci95_high"] = [round(cis[e][1], 3) for e in summary.index]
summary["docs"] = all_df.groupby("engine").size()
print("=== OVERALL (higher field_hit_core is better; lower time / cer / wer is better) ===")
display(summary)

print("=== BY LANGUAGE ===")
display(all_df.groupby(["language", "engine"])[metrics].mean().round(3))

print("=== BY SOURCE (cord = real English receipts, synthetic = generated) ===")
display(all_df.groupby(["source", "engine"])[metrics].mean().round(3))

print("=== BY QUALITY ===")
display(all_df.groupby(["quality", "engine"])[metrics].mean().round(3))

print("=== HIT RATE PER FIELD ===")
display(all_df.groupby("engine")[["hit_" + f for f in FIELDS]].mean().round(3))

print("documents where the engine raised an error:")
print(all_df[all_df["error"].fillna("") != ""].groupby("engine").size())

all_df.to_csv(f"{BENCH_DIR}/all_engines_val.csv", index=False)

by_eng = all_df.groupby("engine")[metrics].mean()
if by_eng["cer"].notna().any():
    suggested = by_eng["cer"].idxmin()
else:
    suggested = by_eng["field_hit_core"].idxmax()
print("\nSuggested winner:", suggested)

documents per engine: {'tesseract': 164, 'easyocr': 164, 'paddle': 164, 'surya': 164}
documents scored by ALL engines: 164
=== OVERALL (higher field_hit_core is better; lower time / cer / wer is better) ===


,cer,wer,field_hit_core,field_hit,time,core_ci95_low,core_ci95_high,docs
engine,,,,,,,,
easyocr,NaN,NaN,0.651,0.630,1.256,0.599,0.703,164
paddle_gpu,NaN,NaN,0.796,0.800,1.096,0.750,0.839,164
surya,NaN,NaN,0.877,0.852,2.453,0.843,0.912,164
tesseract,NaN,NaN,0.468,0.488,3.468,0.412,0.522,164


=== BY LANGUAGE ===


cer  wer  field_hit_core  field_hit   time
language engine                                                
ar       easyocr     NaN  NaN           0.517      0.492  0.951
         paddle_gpu  NaN  NaN           0.550      0.558  0.997
         surya       NaN  NaN           0.733      0.662  2.194
         tesseract   NaN  NaN           0.200      0.217  3.187
en       easyocr     NaN  NaN           0.640      0.625  1.271
         paddle_gpu  NaN  NaN           0.875      0.875  1.267
         surya       NaN  NaN           0.905      0.909  2.837
         tesseract   NaN  NaN           0.572      0.557  4.686
mixed    easyocr     NaN  NaN           0.794      0.771  1.551
         paddle_gpu  NaN  NaN           0.983      0.988  1.070
         surya       NaN  NaN           1.000      1.000  2.430
         tesseract   NaN  NaN           0.661      0.708  2.857

=== BY SOURCE (cord = real English receipts, synthetic = generated) ===


cer  wer  field_hit_core  field_hit   time
source    engine                                                
cord      easyocr     NaN  NaN           0.300      0.300  1.123
          paddle_gpu  NaN  NaN           0.633      0.633  1.758
          surya       NaN  NaN           0.767      0.767  3.630
          tesseract   NaN  NaN           0.300      0.300  8.561
synthetic easyocr     NaN  NaN           0.687      0.663  1.270
          paddle_gpu  NaN  NaN           0.812      0.817  1.030
          surya       NaN  NaN           0.888      0.861  2.334
          tesseract   NaN  NaN           0.485      0.507  2.955

=== BY QUALITY ===


cer  wer  field_hit_core  field_hit   time
quality engine                                                
blurry  easyocr     NaN  NaN           0.496      0.442  1.225
        paddle_gpu  NaN  NaN           0.735      0.750  1.041
        surya       NaN  NaN           0.855      0.833  2.287
        tesseract   NaN  NaN           0.444      0.481  2.622
clear   easyocr     NaN  NaN           0.589      0.600  1.178
        paddle_gpu  NaN  NaN           0.737      0.744  1.263
        surya       NaN  NaN           0.826      0.806  2.700
        tesseract   NaN  NaN           0.419      0.439  4.654
noisy   easyocr     NaN  NaN           0.806      0.778  1.181
        paddle_gpu  NaN  NaN           0.843      0.854  1.003
        surya       NaN  NaN           0.907      0.875  2.359
        tesseract   NaN  NaN           0.574      0.618  3.738
tilted  easyocr     NaN  NaN           0.727      0.705  1.426
        paddle_gpu  NaN  NaN           0.871      0.858  1.050
        surya       NaN  NaN           0.924      0.898  2.424
        tesseract   NaN  NaN           0.455      0.438  2.785

=== HIT RATE PER FIELD ===


,hit_invoice_number,hit_date,hit_supplier,hit_total
engine,,,,
easyocr,0.537,0.591,0.762,0.677
paddle_gpu,0.819,0.832,0.921,0.665
surya,0.752,0.779,0.976,0.902
tesseract,0.530,0.570,0.506,0.390


documents where the engine raised an error:
Series([], dtype: int64)

Suggested winner: surya


### Routing check: one engine for everything, or the best engine per group?
`routed` is optimistic, because the best engine per group is chosen on the same documents it is scored on. Use it only as a hint.
If routed is not clearly better than the best single engine, use the single engine: it is simpler.

In [ ]:
piv = all_df.pivot_table(index=["language", "quality"], columns="engine",
                         values="field_hit_core", aggfunc="mean").round(3)
best = piv.apply(lambda r: r.idxmax() if r.notna().any() else None, axis=1)
display(piv.assign(best=best))

tmp = all_df.copy()
tmp["group_best"] = [best.get((l, q)) for l, q in zip(tmp["language"], tmp["quality"])]
routed = tmp[tmp["engine"] == tmp["group_best"]]["field_hit_core"].mean()
print("best single engine :", suggested, round(float(summary.loc[suggested, "field_hit_core"]), 3))
print("routed (optimistic):", round(float(routed), 3))

engine            easyocr  paddle_gpu  surya  tesseract        best
language quality                                                   
ar       blurry     0.244       0.333  0.622      0.133       surya
         clear      0.578       0.600  0.711      0.222       surya
         noisy      0.644       0.622  0.778      0.222       surya
         tilted     0.600       0.644  0.822      0.222       surya
en       blurry     0.815       1.000  1.000      0.704  paddle_gpu
         clear      0.300       0.633  0.767      0.300       surya
         noisy      0.889       1.000  1.000      1.000  paddle_gpu
         tilted     0.786       1.000  0.952      0.595  paddle_gpu
mixed    blurry     0.556       0.978  1.000      0.600       surya
         clear      0.889       0.978  1.000      0.733       surya
         noisy      0.933       1.000  1.000      0.756  paddle_gpu
         tilted     0.800       0.978  1.000      0.556       surya

best single engine : surya 0.877
routed (optimistic): 0.881


### Pick the winner
Set `WINNER` yourself if you disagree with the suggestion. Check the time estimate: train alone has 2,759 documents.
Surya 0.17 and EasyOCR/Paddle give text **lines**, Tesseract gives **words**. Tell Person 3 which one you used, since it changes how they label tokens.

In [13]:
WINNER = suggested     # or write it yourself: "tesseract" / "easyocr" / "paddle_gpu" / "surya"
ENGINE = ENGINE_OF.get(WINNER, WINNER)
print("Chosen engine:", WINNER)

avg_secs = all_df[all_df.engine == WINNER]["time"].mean()
total_docs = sum(len(load_split(s)) for s in ["train", "val", "test"])
print(f"avg {avg_secs:.2f}s/doc -> about {avg_secs * total_docs / 3600:.1f} hours for {total_docs} documents")
if ENGINE not in ENGINES_TO_TEST:
    print(f"\nNOTE: '{ENGINE}' is not installed in this session (MODE = {MODE}). Run Part B in the session where it was benchmarked.")

Chosen engine: surya
avg 2.45s/doc -> about 3.1 hours for 4500 documents


## Part B - Run the winner on train / val / test
Saved to Drive after every document. If Colab disconnects, run the cells again and it continues where it stopped.

In [14]:
# Reuse the earlier EasyOCR run (saved in /content/ocr_output) if EasyOCR won and that folder still exists.
OLD_DIR = "/content/ocr_output"
if ENGINE == "easyocr" and os.path.isdir(OLD_DIR):
    for s in ["train", "val", "test"]:
        src = f"{OLD_DIR}/{s}_with_ocr.jsonl"
        dst = f"{OUT_DIR}/{s}_with_ocr.jsonl"
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy(src, dst)
            print("reused", src)

In [ ]:
# ===== Part B: run the winner on train / val / test (built to survive Colab disconnects) =====
import gc
from IPython.display import display, Javascript

# 1) Keep-alive: clicks Colab's "Connect" button every minute so the idle timer never expires
#    (only works while this browser tab stays open and the computer does not sleep).
display(Javascript('''
if (!window._ocr_keepalive) {
  window._ocr_keepalive = setInterval(() => {
    const b = document.querySelector("colab-connect-button");
    if (b && b.shadowRoot) { const c = b.shadowRoot.querySelector("#connect"); if (c) c.click(); }
  }, 60000);
}
'''))

SAVE_EVERY = 20          # force the Drive file to disk every N documents
CLEAN_EVERY = 50         # free GPU/RAM every N documents (long runs can crash the session by running out of memory)
MAX_RESTARTS = 50        # how many times a split is restarted after an unexpected error
RETRY_FAILED = True      # True = documents whose saved row has an OCR error are processed again (a .bak copy is kept)


class StopRun(BaseException):
    """Raised to stop the whole run on purpose (not retried by run_all)."""


def cleanup():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def repair_file(path):
    """If the last line was half-written when the session died, cut it off so the file stays valid."""
    if not os.path.exists(path):
        return
    ends, valid, pos = [], [], 0
    with open(path, "rb") as f:
        for raw in f:
            pos += len(raw)
            if not raw.endswith(b"\n"):
                break
            try:
                json.loads(raw)
                valid.append(True)
            except Exception:
                valid.append(False)
            ends.append(pos)
    keep = len(ends)
    while keep and not valid[keep - 1]:
        keep -= 1
    cut = ends[keep - 1] if keep else 0
    size = os.path.getsize(path)
    if cut != size:
        print(f"repair: removed {size - cut} bytes of a half-written last line in {os.path.basename(path)}")
        with open(path, "r+b") as f:
            f.truncate(cut)


def done_ids_in(path):
    ids = set()
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    try:
                        ids.add(json.loads(line)["doc_id"])
                    except Exception:
                        pass
    return ids


def drop_failed(path):
    """Remove saved rows that carry an 'error' so they get processed again. Keeps <file>.bak."""
    if not os.path.exists(path):
        return 0
    kept, dropped, seen = [], 0, set()
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except Exception:
                dropped += 1
                continue
            if (row.get("ocr") or {}).get("error") or row.get("doc_id") in seen:
                dropped += 1                      # failed row, or a duplicate of a document already kept
            else:
                seen.add(row.get("doc_id"))
                kept.append(line if line.endswith("\n") else line + "\n")
    if dropped:
        shutil.copy(path, path + ".bak")
        tmp = path + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            f.writelines(kept)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp, path)
    return dropped


def run_split(split, engine):
    recs = load_split(split)
    out_path = f"{OUT_DIR}/{split}_with_ocr.jsonl"

    repair_file(out_path)
    done = done_ids_in(out_path)
    todo = [r for r in recs if r["doc_id"] not in done]

    print(f"{split}: {len(recs)} total, {len(done)} already done, {len(todo)} to do")
    if not todo:
        return

    ok = bad = 0
    errs = {}
    with open(out_path, "a", encoding="utf-8") as out:
        for n, rec in enumerate(tqdm(todo, desc=f"{engine} {split}"), 1):
            t0 = time.time()
            image_path = resolve_image_path(rec["image_path"])

            if not image_path or not os.path.exists(image_path):
                rec["ocr"] = {"engine": engine, "words": [], "error": "Image not found"}
                bad += 1
            else:
                words, err = None, None
                for attempt in (1, 2):              # one retry after freeing memory
                    try:
                        words, err = run_ocr(engine, image_path), None
                        break
                    except Exception as e:
                        err = e
                        cleanup()
                if err is None:
                    rec["ocr"] = {"engine": engine, "words": words}
                    rec["processing_time"] = round(time.time() - t0, 3)
                    ok += 1
                else:
                    rec["ocr"] = {"engine": engine, "words": [], "error": str(err)[:200]}
                    bad += 1
            if rec["ocr"].get("error"):
                e_ = rec["ocr"]["error"]
                errs[e_] = errs.get(e_, 0) + 1
                if errs[e_] == 1:
                    print(f"\n  first '{e_[:120]}' on {rec['doc_id']} -> {image_path}")

            out.write(json.dumps(rec, ensure_ascii=False) + "\n")
            out.flush()
            if n == 20 and ok == 0:                # nothing worked in the first 20 documents: stop, do not waste hours
                out.flush()
                print("\n!! The first 20 documents all failed. Errors seen:")
                for e_, c_ in sorted(errs.items(), key=lambda x: -x[1])[:5]:
                    print(f"   {c_} x {e_}")
                raise StopRun("fix the error above, then run the cell again")
            if n % SAVE_EVERY == 0:
                os.fsync(out.fileno())
            if n % CLEAN_EVERY == 0:
                cleanup()
        out.flush()
        os.fsync(out.fileno())

    print(f"{split} done. ok={ok} failed={bad} -> {out_path}")
    for e_, c_ in sorted(errs.items(), key=lambda x: -x[1])[:5]:
        print(f"   {c_} x {e_}")


def run_all(engine, splits=("train", "val", "test")):
    for split in splits:
        if RETRY_FAILED:
            repair_file(f"{OUT_DIR}/{split}_with_ocr.jsonl")
            n = drop_failed(f"{OUT_DIR}/{split}_with_ocr.jsonl")
            if n:
                print(f"{split}: {n} failed rows removed, they will be processed again")
        for attempt in range(1, MAX_RESTARTS + 1):
            try:
                run_split(split, engine)
                break
            except Exception as e:                  # Drive hiccup, odd record, etc.: wait and continue from the saved file
                print(f"!! {split} stopped ({type(e).__name__}: {e}) -> restart {attempt}/{MAX_RESTARTS} in 30 s")
                cleanup()
                time.sleep(30)
                if not os.path.isdir(OUT_DIR):
                    print("Drive looks unmounted, mounting again")
                    drive.mount('/content/drive', force_remount=True)
        else:
            print(f"gave up on {split} after {MAX_RESTARTS} restarts")
            return
    print("\nALL SPLITS DONE")


run_all(ENGINE)


<IPython.core.display.Javascript object>

train: 2759 total, 1087 already done, 1672 to do


surya train:   0%|          | 0/1672 [00:00<?, ?it/s]

Exception ignored in: <function _xla_gc_callback at 0x7854de68d080>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/jax/_src/lib/__init__.py", line 125, in _xla_gc_callback
    def _xla_gc_callback(*args):
KeyboardInterrupt: 


KeyboardInterrupt: 

In [17]:
# ===== Part B: run the winner on train / val / test (built to survive Colab disconnects) =====
import gc
from IPython.display import display, Javascript

# 1) Keep-alive: clicks Colab's "Connect" button every minute so the idle timer never expires
#    (only works while this browser tab stays open and the computer does not sleep).
display(Javascript('''
if (!window._ocr_keepalive) {
  window._ocr_keepalive = setInterval(() => {
    const b = document.querySelector("colab-connect-button");
    if (b && b.shadowRoot) { const c = b.shadowRoot.querySelector("#connect"); if (c) c.click(); }
  }, 60000);
}
'''))

SAVE_EVERY = 20          # force the Drive file to disk every N documents
CLEAN_EVERY = 50         # free GPU/RAM every N documents (long runs can crash the session by running out of memory)
MAX_RESTARTS = 50        # how many times a split is restarted after an unexpected error
RETRY_FAILED = True      # True = documents whose saved row has an OCR error are processed again (a .bak copy is kept)


class StopRun(BaseException):
    """Raised to stop the whole run on purpose (not retried by run_all)."""


def cleanup():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def repair_file(path):
    """If the last line was half-written when the session died, cut it off so the file stays valid."""
    if not os.path.exists(path):
        return
    ends, valid, pos = [], [], 0
    with open(path, "rb") as f:
        for raw in f:
            pos += len(raw)
            if not raw.endswith(b"\n"):
                break
            try:
                json.loads(raw)
                valid.append(True)
            except Exception:
                valid.append(False)
            ends.append(pos)
    keep = len(ends)
    while keep and not valid[keep - 1]:
        keep -= 1
    cut = ends[keep - 1] if keep else 0
    size = os.path.getsize(path)
    if cut != size:
        print(f"repair: removed {size - cut} bytes of a half-written last line in {os.path.basename(path)}")
        with open(path, "r+b") as f:
            f.truncate(cut)


def done_ids_in(path):
    ids = set()
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    try:
                        ids.add(json.loads(line)["doc_id"])
                    except Exception:
                        pass
    return ids


def drop_failed(path):
    """Remove saved rows that carry an 'error' so they get processed again. Keeps <file>.bak."""
    if not os.path.exists(path):
        return 0
    kept, dropped, seen = [], 0, set()
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except Exception:
                dropped += 1
                continue
            if (row.get("ocr") or {}).get("error") or row.get("doc_id") in seen:
                dropped += 1                      # failed row, or a duplicate of a document already kept
            else:
                seen.add(row.get("doc_id"))
                kept.append(line if line.endswith("\n") else line + "\n")
    if dropped:
        shutil.copy(path, path + ".bak")
        tmp = path + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            f.writelines(kept)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp, path)
    return dropped


def run_split(split, engine):
    recs = load_split(split)
    out_path = f"{OUT_DIR}/{split}_with_ocr.jsonl"

    repair_file(out_path)
    done = done_ids_in(out_path)
    todo = [r for r in recs if r["doc_id"] not in done]

    print(f"{split}: {len(recs)} total, {len(done)} already done, {len(todo)} to do")
    if not todo:
        return

    ok = bad = 0
    errs = {}
    with open(out_path, "a", encoding="utf-8") as out:
        for n, rec in enumerate(tqdm(todo, desc=f"{engine} {split}"), 1):
            t0 = time.time()
            image_path = resolve_image_path(rec["image_path"])

            if not image_path or not os.path.exists(image_path):
                rec["ocr"] = {"engine": engine, "words": [], "error": "Image not found"}
                bad += 1
            else:
                words, err = None, None
                for attempt in (1, 2):              # one retry after freeing memory
                    try:
                        words, err = run_ocr(engine, image_path), None
                        break
                    except Exception as e:
                        err = e
                        cleanup()
                if err is None:
                    rec["ocr"] = {"engine": engine, "words": words}
                    rec["processing_time"] = round(time.time() - t0, 3)
                    ok += 1
                else:
                    rec["ocr"] = {"engine": engine, "words": [], "error": str(err)[:200]}
                    bad += 1
            if rec["ocr"].get("error"):
                e_ = rec["ocr"]["error"]
                errs[e_] = errs.get(e_, 0) + 1
                if errs[e_] == 1:
                    print(f"\n  first '{e_[:120]}' on {rec['doc_id']} -> {image_path}")

            out.write(json.dumps(rec, ensure_ascii=False) + "\n")
            out.flush()
            if n == 20 and ok == 0:                # nothing worked in the first 20 documents: stop, do not waste hours
                out.flush()
                print("\n!! The first 20 documents all failed. Errors seen:")
                for e_, c_ in sorted(errs.items(), key=lambda x: -x[1])[:5]:
                    print(f"   {c_} x {e_}")
                raise StopRun("fix the error above, then run the cell again")
            if n % SAVE_EVERY == 0:
                os.fsync(out.fileno())
            if n % CLEAN_EVERY == 0:
                cleanup()
        out.flush()
        os.fsync(out.fileno())

    print(f"{split} done. ok={ok} failed={bad} -> {out_path}")
    for e_, c_ in sorted(errs.items(), key=lambda x: -x[1])[:5]:
        print(f"   {c_} x {e_}")


def run_all(engine, splits=("train", "val", "test")):
    for split in splits:
        if RETRY_FAILED:
            repair_file(f"{OUT_DIR}/{split}_with_ocr.jsonl")
            n = drop_failed(f"{OUT_DIR}/{split}_with_ocr.jsonl")
            if n:
                print(f"{split}: {n} failed rows removed, they will be processed again")
        for attempt in range(1, MAX_RESTARTS + 1):
            try:
                run_split(split, engine)
                break
            except Exception as e:                  # Drive hiccup, odd record, etc.: wait and continue from the saved file
                print(f"!! {split} stopped ({type(e).__name__}: {e}) -> restart {attempt}/{MAX_RESTARTS} in 30 s")
                cleanup()
                time.sleep(30)
                if not os.path.isdir(OUT_DIR):
                    print("Drive looks unmounted, mounting again")
                    drive.mount('/content/drive', force_remount=True)
        else:
            print(f"gave up on {split} after {MAX_RESTARTS} restarts")
            return
    print("\nALL SPLITS DONE")


run_all(ENGINE, ("train",))      # train split only

<IPython.core.display.Javascript object>

train: 2759 total, 2759 already done, 0 to do

ALL SPLITS DONE


In [19]:
# ===== STRICT CHECK of one split's OCR output (read-only, changes nothing) =====
import json, os, time
from collections import Counter

SPLIT = "val"
try:
    OUT_DIR, SPLITS_DIR
except NameError:
    BASE = "/content/drive/MyDrive/OCR dataset v1"
    OUT_DIR, SPLITS_DIR = BASE + "/ocr_output", BASE + "/data/splits"

out_path, split_path = f"{OUT_DIR}/{SPLIT}_with_ocr.jsonl", f"{SPLITS_DIR}/{SPLIT}.jsonl"
rows, bad_lines = [], 0
with open(out_path, encoding="utf-8") as f:
    for l in f:
        if l.strip():
            try: rows.append(json.loads(l))
            except Exception: bad_lines += 1
want = {}
with open(split_path, encoding="utf-8") as f:
    for l in f:
        if l.strip():
            r = json.loads(l); want[r["doc_id"]] = r

ids = [r.get("doc_id") for r in rows]
uniq = set(ids)
problems, warnings = [], []

def word_ok(w):
    b = w.get("bbox")
    return (isinstance(w.get("text"), str) and w["text"].strip() != "" and isinstance(b, list) and len(b) == 4
            and b[2] >= b[0] and b[3] >= b[1] and isinstance(w.get("conf"), (int, float)) and 0 <= w["conf"] <= 1)

errors = [r for r in rows if (r.get("ocr") or {}).get("error")]
empty = [r for r in rows if not (r.get("ocr") or {}).get("error") and not (r.get("ocr") or {}).get("words")]
bad_words = [r for r in rows if any(not word_ok(w) for w in (r.get("ocr") or {}).get("words", []))]
no_time = [r for r in rows if not (r.get("ocr") or {}).get("error") and "processing_time" not in r]
engines = Counter((r.get("ocr") or {}).get("engine") for r in rows)
missing, extra = set(want) - uniq, uniq - set(want)

print(f"file: {out_path}")
print(f"last saved: {time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(out_path)))} | size {os.path.getsize(out_path)/1e6:.1f} MB")
print(f"expected {len(want)} | rows {len(rows)} | unique {len(uniq)} | duplicates {len(ids) - len(uniq)} | unreadable lines {bad_lines}")
print(f"missing {len(missing)} | not in split {len(extra)} | errors {len(errors)} | empty OCR {len(empty)} | malformed words {len(bad_words)}")
print(f"engines {dict(engines)} | rows without processing_time {len(no_time)}")

if bad_lines: problems.append(f"{bad_lines} unreadable lines")
if len(rows) != len(uniq): problems.append(f"{len(rows) - len(uniq)} duplicate rows")
if missing: problems.append(f"{len(missing)} documents missing, e.g. {sorted(missing)[:3]}")
if extra: problems.append(f"{len(extra)} documents that are not in the split, e.g. {sorted(extra)[:3]}")
if errors: problems.append(f"{len(errors)} rows with errors: {Counter(r['ocr']['error'][:80] for r in errors).most_common(2)}")
if bad_words: problems.append(f"{len(bad_words)} rows with malformed words (empty text / bad box / bad conf)")
if len([e for e in engines if e]) > 1: warnings.append("more than one engine produced these rows")
if no_time: warnings.append(f"{len(no_time)} rows have no processing_time (written by an older version)")
if len(empty) > 0.02 * max(len(rows), 1): warnings.append(f"{len(empty)} rows have no words ({100*len(empty)/len(rows):.1f}%): check a few images")

# per group breakdown, if the split has these fields
for key in ("language", "quality", "source"):
    if any(key in v for v in want.values()):
        g = Counter(); ok = Counter()
        for r in rows:
            k = (want.get(r.get("doc_id")) or {}).get(key)
            g[k] += 1; ok[k] += bool((r.get("ocr") or {}).get("words")) and not (r.get("ocr") or {}).get("error")
        print(f"by {key}: " + " | ".join(f"{k}: {ok[k]}/{g[k]} ok" for k in sorted(g, key=str)))

# average words per document is a quick sanity check
wc = [len(r["ocr"]["words"]) for r in rows if not r["ocr"].get("error")]
if wc: print(f"words per document: min {min(wc)} | median {sorted(wc)[len(wc)//2]} | max {max(wc)}")

print("\n" + "=" * 60)
for p in problems: print("PROBLEM:", p)
for w in warnings: print("warning:", w)
print("VERDICT:", f"{SPLIT.upper()} IS FULLY DONE AND CORRECT" if not problems else f"{SPLIT.upper()} IS NOT CORRECT YET (see PROBLEM lines)")

file: /content/drive/MyDrive/OCR dataset v1/ocr_output/val_with_ocr.jsonl
last saved: 2026-11-14 10:07 | size 1.5 MB
expected 516 | rows 516 | unique 516 | duplicates 0 | unreadable lines 0
missing 0 | not in split 0 | errors 0 | empty OCR 0 | malformed words 0
engines {'surya': 516} | rows without processing_time 0
by language: ar: 197/197 ok | en: 200/200 ok | mixed: 119/119 ok
by quality: blurry: 74/74 ok | clear: 283/283 ok | noisy: 78/78 ok | tilted: 81/81 ok
by source: cord: 149/149 ok | synthetic: 367/367 ok
words per document: min 3 | median 26 | max 60

VERDICT: VAL IS FULLY DONE AND CORRECT


In [24]:
# ===== VERY STRICT CHECK of one split's OCR output (read-only; RERUN_SAMPLE > 0 also runs OCR on a few docs) =====
import json, os, re, time, random, unicodedata
from collections import Counter, defaultdict
from tqdm.auto import tqdm

SPLIT = "test"
CHECK_IMAGES = True      # open every image header: file exists, is readable, and every box fits inside the image
RERUN_SAMPLE = 0         # e.g. 30: re-run OCR on 30 random documents and compare with the saved result (needs the engine loaded)

try:
    OUT_DIR, SPLITS_DIR
except NameError:
    BASE = "/content/drive/MyDrive/OCR dataset v1"
    OUT_DIR, SPLITS_DIR = BASE + "/ocr_output", BASE + "/data/splits"
try:
    DATA_DIR
except NameError:
    DATA_DIR = os.path.dirname(SPLITS_DIR)

def _resolve(p):
    f = globals().get("resolve_image_path")
    if f:
        return f(p)
    if os.path.exists(p):
        return p
    return os.path.join(DATA_DIR, p[5:] if p.startswith("data/") else p)

_AR = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")
def _norm(s):
    s = unicodedata.normalize("NFKC", str(s or "")).translate(_AR)
    s = re.sub("[ً-ٰٟـ]", "", s)
    s = re.sub("[آإأ]", "ا", s).replace("ى", "ي").replace("ة", "ه")
    return re.sub(r"[\W_]+", "", s.lower())

def _read(path):
    rows, bad = [], 0
    with open(path, encoding="utf-8") as f:
        for l in f:
            if l.strip():
                try: rows.append(json.loads(l))
                except Exception: bad += 1
    return rows, bad

out_path, split_path = f"{OUT_DIR}/{SPLIT}_with_ocr.jsonl", f"{SPLITS_DIR}/{SPLIT}.jsonl"
rows, bad_lines = _read(out_path)
want = {r["doc_id"]: r for r in _read(split_path)[0]}
ids = [r.get("doc_id") for r in rows]
uniq = set(ids)
problems, warnings = [], []
raw = open(out_path, "rb").read()

def word_ok(w):
    b = w.get("bbox")
    return (isinstance(w.get("text"), str) and w["text"].strip() != "" and isinstance(b, list) and len(b) == 4
            and all(isinstance(v, (int, float)) for v in b) and b[2] >= b[0] and b[3] >= b[1]
            and isinstance(w.get("conf"), (int, float)) and 0 <= w["conf"] <= 1)

# ---------- 1. structure (same as the normal check) ----------
errors = [r for r in rows if (r.get("ocr") or {}).get("error")]
empty = [r for r in rows if not (r.get("ocr") or {}).get("error") and not (r.get("ocr") or {}).get("words")]
bad_words = [r for r in rows if any(not word_ok(w) for w in (r.get("ocr") or {}).get("words", []))]
engines = Counter((r.get("ocr") or {}).get("engine") for r in rows)
missing, extra = set(want) - uniq, uniq - set(want)
print(f"file: {out_path}\nlast saved: {time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(out_path)))} | {os.path.getsize(out_path)/1e6:.1f} MB")
print(f"expected {len(want)} | rows {len(rows)} | unique {len(uniq)} | missing {len(missing)} | not in split {len(extra)} | "
      f"errors {len(errors)} | empty {len(empty)} | malformed words {len(bad_words)}")
if bad_lines: problems.append(f"{bad_lines} unreadable lines")
if len(rows) != len(uniq): problems.append(f"{len(rows) - len(uniq)} duplicate rows")
if missing: problems.append(f"{len(missing)} documents missing, e.g. {sorted(missing)[:3]}")
if extra: problems.append(f"{len(extra)} documents not in the split, e.g. {sorted(extra)[:3]}")
if errors: problems.append(f"{len(errors)} rows with errors")
if empty: problems.append(f"{len(empty)} rows with no words (strict mode: every document should have text)")
if bad_words: problems.append(f"{len(bad_words)} rows with malformed words")
if len([e for e in engines if e]) != 1: problems.append(f"expected exactly one engine, found {dict(engines)}")
if b"\x00" in raw: problems.append("the file contains NUL bytes (half-written or corrupted block)")
if not raw.endswith(b"\n"): problems.append("the file does not end with a newline (last line may be cut)")

# ---------- 2. the saved row still matches the split record ----------
meta_bad = Counter(); meta_docs = set()
for r in rows:
    w = want.get(r.get("doc_id"))
    if not w: continue
    for k, v in w.items():
        if k in ("ocr", "processing_time"): continue
        if r.get(k) != v:
            meta_bad[k] += 1; meta_docs.add(r["doc_id"])
print(f"rows whose copied fields differ from the split record: {len(meta_docs)}" + (f" {dict(meta_bad)}" if meta_bad else ""))
if meta_docs: problems.append(f"{len(meta_docs)} rows changed vs the split (fields {dict(meta_bad)}), e.g. {sorted(meta_docs)[:3]}")

# ---------- 3. identical OCR on different documents (stale / copied result) ----------
sig = defaultdict(list)
for r in rows:
    ws = (r.get("ocr") or {}).get("words") or []
    if len(ws) >= 3:
        sig[hash(tuple((w["text"], tuple(w["bbox"])) for w in ws))].append(r["doc_id"])
same = [v for v in sig.values() if len(set(v)) > 1]
print(f"groups of different documents with IDENTICAL OCR output: {len(same)}")
if same: problems.append(f"{len(same)} groups of different documents share identical OCR text+boxes, e.g. {same[0][:3]}")

# ---------- 4. images exist and the boxes fit inside them ----------
if CHECK_IMAGES:
    from PIL import Image
    miss_img, unread, out_box = [], [], []
    for r in tqdm(rows, desc="images"):
        p = _resolve(r.get("image_path", ""))
        if not p or not os.path.exists(p): miss_img.append(r["doc_id"]); continue
        try:
            with Image.open(p) as im: W, H = im.size
        except Exception:
            unread.append(r["doc_id"]); continue
        for w in (r.get("ocr") or {}).get("words", []):
            b = w["bbox"]
            if b[0] < -0.02 * W or b[1] < -0.02 * H or b[2] > 1.02 * W or b[3] > 1.02 * H:
                out_box.append(r["doc_id"]); break
    print(f"image missing {len(miss_img)} | unreadable {len(unread)} | docs with a box outside the image {len(out_box)}")
    if miss_img: problems.append(f"{len(miss_img)} image files not found, e.g. {miss_img[:3]}")
    if unread: problems.append(f"{len(unread)} image files unreadable, e.g. {unread[:3]}")
    if out_box:
        msg = f"{len(out_box)} docs have a box outside the image, e.g. {out_box[:3]}"
        if len(out_box) > 0.02 * len(rows):
            problems.append(msg)
        else:
            warnings.append(msg)

# ---------- 5. does the OCR text contain the labelled fields? (information, not pass/fail) ----------
hit_by = defaultdict(lambda: [0, 0]); worst = []
tot_h = tot_n = 0
for r in rows:
    w = want.get(r.get("doc_id")) or {}
    gt = w.get("ground_truth") or {}
    txt = _norm(" ".join(x["text"] for x in (r.get("ocr") or {}).get("words", [])))
    hs = [(_norm(gt[k]) in txt) for k in ("invoice_number", "supplier", "total") if gt.get(k) not in (None, "") and _norm(gt[k])]
    if not hs: continue
    h = sum(hs) / len(hs); tot_h += h; tot_n += 1
    for key in ("language", "quality", "source"):
        if key in w: hit_by[(key, w[key])][0] += h; hit_by[(key, w[key])][1] += 1
    if h == 0: worst.append(r["doc_id"])
if tot_n:
    print(f"labelled fields found in the OCR text: {100*tot_h/tot_n:.1f}% over {tot_n} documents | documents with 0 fields found: {len(worst)}")
    for key in ("language", "quality", "source"):
        parts = [f"{v}: {100*s/n:.0f}%" for (k, v), (s, n) in sorted(hit_by.items(), key=lambda x: str(x[0])) if k == key]
        if parts: print(f"   by {key}: " + " | ".join(parts))
    if worst: print(f"   examples with 0 fields found (open these images): {worst[:5]}")

# ---------- 6. spot re-run: does OCR on the image give the saved text again? ----------
if RERUN_SAMPLE:
    fn = globals().get("run_ocr"); eng = globals().get("ENGINE")
    if not fn or not eng:
        warnings.append("RERUN_SAMPLE needs run_ocr and ENGINE in this session (run the setup + engines cells first)")
    else:
        random.seed(1); low = []
        for r in tqdm(random.sample(rows, min(RERUN_SAMPLE, len(rows))), desc="re-run"):
            new = {_norm(w["text"]) for w in fn(eng, _resolve(r["image_path"]))}
            old = {_norm(w["text"]) for w in r["ocr"]["words"]}
            j = len(new & old) / max(len(new | old), 1)
            if j < 0.8: low.append((r["doc_id"], round(j, 2)))
        print(f"re-run sample: {len(low)} of {min(RERUN_SAMPLE, len(rows))} documents differ from the saved result {low[:5]}")
        if low: problems.append(f"{len(low)} re-run documents do not match their saved OCR (similarity < 0.8)")

print("\n" + "=" * 60)
for p in problems: print("PROBLEM:", p)
for w in warnings: print("warning:", w)
print("VERDICT:", f"{SPLIT.upper()} PASSES THE STRICT CHECK" if not problems else f"{SPLIT.upper()} FAILS THE STRICT CHECK (see PROBLEM lines)")

file: /content/drive/MyDrive/OCR dataset v1/ocr_output/test_with_ocr.jsonl
last saved: 2026-11-14 11:24 | 5.0 MB
expected 1225 | rows 1225 | unique 1225 | missing 0 | not in split 0 | errors 0 | empty 0 | malformed words 0
rows whose copied fields differ from the split record: 0
groups of different documents with IDENTICAL OCR output: 0


images:   0%|          | 0/1225 [00:00<?, ?it/s]

image missing 0 | unreadable 0 | docs with a box outside the image 0
labelled fields found in the OCR text: 77.4% over 1225 documents | documents with 0 fields found: 4
   by language: ar: 64% | en: 78% | mixed: 99%
   by quality: blurry: 75% | clear: 79% | noisy: 77% | tilted: 75%
   by source: cord: 71% | synthetic: 78%
   examples with 0 fields found (open these images): ['cord_train_00775', 'cord_train_00795', 'cord_train_00071', 'cord_train_00240']

VERDICT: TEST PASSES THE STRICT CHECK


In [22]:
import hashlib, json, os
from collections import defaultdict

SPLIT = "train"
rows = [json.loads(l) for l in open(f"{OUT_DIR}/{SPLIT}_with_ocr.jsonl", encoding="utf-8") if l.strip()]

def sig(r):
    ws = (r.get("ocr") or {}).get("words") or []
    return hash(tuple((w["text"], tuple(w["bbox"])) for w in ws)) if len(ws) >= 3 else None

def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

groups = defaultdict(list)
for r in rows:
    s = sig(r)
    if s is not None:
        groups[s].append(r)
groups = [g for g in groups.values() if len({r["doc_id"] for r in g}) > 1]
print(len(groups), "groups with identical OCR\n")

same_image = 0
for g in groups:
    hashes = {r["doc_id"]: md5(resolve_image_path(r["image_path"])) for r in g}
    identical = len(set(hashes.values())) == 1
    same_image += identical
    print("IDENTICAL IMAGE FILES" if identical else "DIFFERENT IMAGES, same OCR  <-- look at these")
    for r in g:
        print("   ", r["doc_id"], "|", r["image_path"])
print(f"\n{same_image} of {len(groups)} groups are the same image saved under two names")

# do the same OCR outputs also appear in the other splits? (cheap, no images needed)
mine = {sig(r) for g in groups for r in g}
for other in ("val", "test"):
    p = f"{OUT_DIR}/{other}_with_ocr.jsonl"
    if os.path.exists(p):
        o = [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
        hits = [r["doc_id"] for r in o if sig(r) in mine]
        print(f"{other}: {len(hits)} documents have the same OCR as a duplicated train document", hits[:5])

6 groups with identical OCR

DIFFERENT IMAGES, same OCR  <-- look at these
    cord_train_00195 | data/processed/images/cord_train_00195.jpg
    cord_train_00641 | data/processed/images/cord_train_00641.jpg
DIFFERENT IMAGES, same OCR  <-- look at these
    cord_train_00221 | data/processed/images/cord_train_00221.jpg
    cord_validation_00006 | data/processed/images/cord_validation_00006.jpg
DIFFERENT IMAGES, same OCR  <-- look at these
    cord_train_00234 | data/processed/images/cord_train_00234.jpg
    cord_validation_00030 | data/processed/images/cord_validation_00030.jpg
DIFFERENT IMAGES, same OCR  <-- look at these
    cord_train_00438 | data/processed/images/cord_train_00438.jpg
    cord_train_00704 | data/processed/images/cord_train_00704.jpg
DIFFERENT IMAGES, same OCR  <-- look at these
    cord_validation_00027 | data/processed/images/cord_validation_00027.jpg
    cord_validation_00044 | data/processed/images/cord_validation_00044.jpg
IDENTICAL IMAGE FILES
    cord_train_0026

In [ ]:
import json
import os

for split in ["train", "val", "test"]:
    path = f"{OUT_DIR}/{split}_with_ocr.jsonl"

    total = 0
    successful = 0
    failed = 0
    empty = 0

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            total += 1

            error = row.get("ocr", {}).get("error")
            words = row.get("ocr", {}).get("words", [])

            if error:
                failed += 1
            elif words:
                successful += 1
            else:
                empty += 1

    print(
        f"{split}: total={total}, "
        f"successful={successful}, failed={failed}, empty={empty}"
    )

train: total=1087, successful=1087, failed=0, empty=0
val: total=231, successful=0, failed=231, empty=0
test: total=3675, successful=0, failed=3675, empty=0


In [ ]:
import json
from collections import Counter

for split in ["train", "val", "test"]:
    path = f"{OUT_DIR}/{split}_with_ocr.jsonl"
    errors = Counter()

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            error = row.get("ocr", {}).get("error")
            if error:
                errors[str(error)] += 1

    print(f"\n===== {split.upper()} ERRORS =====")
    for error, count in errors.most_common(10):
        print(f"{count}: {error}")


===== TRAIN ERRORS =====

===== VAL ERRORS =====
231: operator torchvision::nms does not exist

===== TEST ERRORS =====
1225: No module named 'surya'
1225: 'SuryaDecoderConfig' object has no attribute 'pad_token_id'
1225: cannot identify image file '/content/drive/MyDrive/OCR dataset v1/ocr_output/train_with_ocr.jsonl'


In [ ]:
import transformers
import surya

print("Transformers:", transformers.__version__)
print("Surya imported successfully")

Transformers: 4.57.3
Surya imported successfully


In [ ]:
import json
import os

path = f"{OUT_DIR}/train_with_ocr.jsonl"

shown = 0
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)

        if row.get("ocr", {}).get("error") == "Image not found":
            print("doc_id:", row.get("doc_id"))
            print("image_path:", row.get("image_path"))
            print("exists:", os.path.exists(row.get("image_path", "")))
            print()
            shown += 1

        if shown == 5:
            break

In [ ]:
test_path = resolve_image_path(
    "data/synth/out/images/synthetic_train_03163.jpg"
)

words = run_ocr(ENGINE, test_path)
print(words[:10])

KeyboardInterrupt: 

In [23]:
# ===== STATUS: what has been reached so far (safe to run any time, changes nothing) =====
import json, os, time
from collections import Counter

try:
    OUT_DIR, SPLITS_DIR
except NameError:
    BASE = "/content/drive/MyDrive/OCR dataset v1"
    OUT_DIR, SPLITS_DIR = BASE + "/ocr_output", BASE + "/data/splits"

def read_jsonl(p):
    rows = []
    if os.path.exists(p):
        with open(p, encoding="utf-8") as f:
            for l in f:
                if l.strip():
                    try: rows.append(json.loads(l))
                    except Exception: pass
    return rows

rank = {"ok": 2, "empty": 1, "error": 0}
all_good = True
print(f"{'split':6}{'expected':>9}{'ok':>7}{'empty':>7}{'failed':>8}{'missing':>9}{'dupes':>7}{'done %':>8}   last saved")
details = []
for split in ["train", "val", "test"]:
    want = {r["doc_id"] for r in read_jsonl(f"{SPLITS_DIR}/{split}.jsonl")}
    path = f"{OUT_DIR}/{split}_with_ocr.jsonl"
    rows = read_jsonl(path)
    best, dupes = {}, 0
    for r in rows:
        o = r.get("ocr") or {}
        st = "error" if o.get("error") else ("ok" if o.get("words") else "empty")
        d = r.get("doc_id")
        if d in best: dupes += 1
        if d not in best or rank[st] > rank[best[d][0]]:
            best[d] = (st, r)
    best = {d: v for d, v in best.items() if d in want}
    c = Counter(v[0] for v in best.values())
    missing = len(want - set(best))
    done = c["ok"] + c["empty"]
    pct = 100 * done / max(len(want), 1)
    last = time.strftime("%m-%d %H:%M", time.localtime(os.path.getmtime(path))) if os.path.exists(path) else "no file"
    print(f"{split:6}{len(want):>9}{c['ok']:>7}{c['empty']:>7}{c['error']:>8}{missing:>9}{dupes:>7}{pct:>7.1f}%   {last}")
    errs = Counter((v[1]["ocr"].get("error") or "")[:110] for v in best.values() if v[0] == "error")
    engines = Counter((v[1].get("ocr") or {}).get("engine") for v in best.values() if v[0] != "error")
    times = [v[1].get("processing_time") for v in best.values() if v[0] == "ok" and v[1].get("processing_time")]
    details.append((split, errs, engines, times))
    all_good &= (missing == 0 and c["error"] == 0 and dupes == 0)

print()
for split, errs, engines, times in details:
    line = f"{split}: engine={dict(engines) or '-'}"
    if times: line += f" | avg {sum(times)/len(times):.2f} s/doc"
    print(line)
    for e, n in errs.most_common(3):
        print(f"   {n} x {e}")

print("\nVERDICT:", "ALL DONE and clean" if all_good else
      "NOT finished: run the run_split cell (it redoes failed/missing documents), then run this cell again")

split  expected     ok  empty  failed  missing  dupes  done %   last saved
train      2759   2759      0       0        0      0  100.0%   11-14 10:17
val         516    516      0       0        0      0  100.0%   11-14 10:07
test       1225   1225      0       0        0      0  100.0%   11-14 11:24

train: engine={'surya': 2759} | avg 6.73 s/doc
val: engine={'surya': 516} | avg 1.95 s/doc
test: engine={'surya': 1225} | avg 2.31 s/doc

VERDICT: ALL DONE and clean


In [25]:
# Final check
for split in ["train", "val", "test"]:
    path = f"{OUT_DIR}/{split}_with_ocr.jsonl"
    recs = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    n_err = sum(1 for r in recs if r["ocr"].get("error"))
    n_empty = sum(1 for r in recs if not r["ocr"]["words"])
    print(f"{split}: {len(recs)} records | errors: {n_err} | empty OCR: {n_empty} | expected: {len(load_split(split))}")
print("\nSample words:")
print(json.dumps(recs[0]["ocr"]["words"][:3], ensure_ascii=False, indent=1))
print("\nFiles for Person 3 / Person 4 are in:", OUT_DIR)

train: 2759 records | errors: 0 | empty OCR: 0 | expected: 2759
val: 516 records | errors: 0 | empty OCR: 0 | expected: 516
test: 1225 records | errors: 0 | empty OCR: 0 | expected: 1225

Sample words:
[
 {
  "text": "مؤسسة الرائد",
  "bbox": [
   612,
   62,
   746,
   88
  ],
  "conf": 0.9996606281825474
 },
 {
  "text": "فاتورة تجارية",
  "bbox": [
   108,
   65,
   183,
   83
  ],
  "conf": 0.999689535299937
 },
 {
  "text": "رقم الفاتورة",
  "bbox": [
   704,
   165,
   756,
   181
  ],
  "conf": 0.9991283714771271
 }
]

Files for Person 3 / Person 4 are in: /content/drive/MyDrive/OCR dataset v1/ocr_output


##**Final Output:**
https://drive.google.com/drive/folders/1FESxKuDn_RZ_A0YNeP-ToP4tLz8IemuO?usp=sharing
